# 01 – Data Preparation

This notebook loads raw node features, node2vec embeddings, and struc2vec embeddings
for a chosen dataset, combines them into a single DataFrame, and saves it to `data/04_models/`
for use in notebooks 02–04.

## Datasets available
| Name | Description |
|------|-------------|
| `facebook` | Facebook ego-graph (default) |
| `amazon` | Amazon co-purchase network |
| `grid` | Power grid network |
| `lastfm` | LastFM social graph |
| `reddit` | Reddit post network |
| `twitch` | Twitch social network |
| `x3` … `x7` | Synthetic ABCD+o graphs |

## What are the features?
Each node has two groups of features:

**Community-free graph features** (computed without community information):
- `lcc` – local clustering coefficient
- `bc` – betweenness centrality
- `cc` – closeness centrality
- `dc` – degree centrality
- `ndc` – normalised degree centrality
- `ec` – eigenvector centrality
- `eccen` – eccentricity
- `core` – core number (k-core decomposition)

**Community-aware features** (require knowledge of communities – used as "cheating" baseline):
- `participation` – participation coefficient (how spread are node's edges across communities)
- `in_mod_deg` – intra-community degree

**Regression targets** (13 continuous community measures):
`cada`, `edge_contr`, `in_mod_deg`, `participation`, `beta_star`,
`l1`, `l2`, `kl`, `hd`, `l12`, `l22`, `kl2`, `hd2`

**Classification target**: `target` (binary: 1 = overlapping/community node, 0 = otherwise)

In [1]:
# ── CHANGE THIS to switch dataset ──────────────────────────────────────────
DATASET = "facebook"
# ───────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.impute import SimpleImputer

ROOT = Path("..")

# Paths derived from dataset name
PATH_FEATURES  = ROOT / f"data/01_raw/data/{DATASET}.parquet"
PATH_EDGES     = ROOT / f"data/01_raw/edges/edge_{DATASET}.parquet"
PATH_N2V       = ROOT / f"data/03_embeddings/{DATASET}_node2vec_embedding.parquet"
PATH_S2V       = ROOT / f"struc2vec/emb/{DATASET}.emb"
PATH_OUTPUT    = ROOT / f"data/04_models/{DATASET}_combined.parquet"

print(f"Dataset: {DATASET}")
print(f"Features : {PATH_FEATURES}")
print(f"Edges    : {PATH_EDGES}")
print(f"Node2Vec : {PATH_N2V}")
print(f"Struc2Vec: {PATH_S2V}")

Dataset: facebook
Features : ../data/01_raw/data/facebook.parquet
Edges    : ../data/01_raw/edges/edge_facebook.parquet
Node2Vec : ../data/03_embeddings/facebook_node2vec_embedding.parquet
Struc2Vec: ../struc2vec/emb/facebook.emb


## Step 1 – Load node features

In [2]:
features = pd.read_parquet(PATH_FEATURES)

print(f"Shape: {features.shape}")
print(f"Columns: {features.columns.tolist()}")
features.head()

Shape: (22470, 23)
Columns: ['node_id', 'target', 'participation', 'in_mod_deg', 'beta_star', 'edge_contr', 'cada', 'kl', 'l1', 'l2', 'hd', 'kl2', 'l12', 'l22', 'hd2', 'lcc', 'bc', 'dc', 'ndc', 'cc', 'ec', 'eccen', 'core']


,node_id,target,participation,in_mod_deg,beta_star,edge_contr,cada,kl,l1,l2,...,l22,hd2,lcc,bc,dc,ndc,cc,ec,eccen,core
0,1,False,1.000000,-0.784970,0.996174,1.000000,1.000000,4.436440,1.976324,1.016336,...,0.679263,0.740483,0.000000,0.000000,1,51.000000,0.182482,1.127609e-07,11,1
1,2,False,0.783737,0.283540,0.826617,0.882353,1.133333,2.996689,1.746428,0.881411,...,0.618068,0.667733,0.481283,0.000153,34,140.529412,0.257752,7.947068e-03,9,28
2,3,False,1.000000,0.923698,0.980070,1.000000,1.000000,3.090420,1.909034,0.982675,...,0.938501,0.828561,0.651515,0.000003,12,15.500000,0.189540,2.584898e-06,11,8
3,4,False,0.820000,-0.206060,0.896488,0.900000,1.111111,5.101546,1.977837,0.931507,...,0.795555,0.788387,0.511111,0.000052,10,29.900000,0.219007,1.553212e-05,10,9
4,5,True,0.961553,1.093894,0.898855,0.980392,1.020000,2.917455,1.862698,0.959152,...,0.948889,0.812959,0.420392,0.000125,51,49.941176,0.230387,1.143732e-04,10,23


## Step 2 – Load and inspect edges

The edge list has two columns: `in` and `out` (source and target node IDs).
We normalise node IDs to be 0-based (Julia uses 1-based indexing).

In [3]:
edges = pd.read_parquet(PATH_EDGES)

# Normalise to 0-based if needed
if 0 not in set(edges["in"]) and 0 not in set(edges["out"]):
    print("Edge IDs start from 1 → converting to 0-based")
    edges = edges - 1
else:
    print("Edge IDs already 0-based")

print(f"Edges shape: {edges.shape}")
print(f"Node ID range: [{edges.min().min()} … {edges.max().max()}]")
edges.head()

Edge IDs start from 1 → converting to 0-based
Edges shape: (170823, 2)
Node ID range: [0 … 22469]


,in,out
0,0,16494
1,1,2788
2,1,3653
3,1,3677
4,1,4902


## Step 3 – Load node2vec embeddings

Node2Vec maps each node to a 16-dimensional vector by simulating biased random walks
and training a Word2Vec model on those walks.
Parameters used: `dim=16`, `walk_length=50`, `window=10`, `p=1`, `q=1`.

The embeddings were pre-computed with `fastnode2vec` and stored as parquet.
Row index = node ID.

In [4]:
n2v = pd.read_parquet(PATH_N2V)

print(f"Shape: {n2v.shape}  (nodes × embedding dimensions)")
print(f"Columns: {n2v.columns.tolist()}")
n2v.head(3)

Shape: (22470, 16)  (nodes × embedding dimensions)
Columns: ['emb_node2vec_0', 'emb_node2vec_1', 'emb_node2vec_2', 'emb_node2vec_3', 'emb_node2vec_4', 'emb_node2vec_5', 'emb_node2vec_6', 'emb_node2vec_7', 'emb_node2vec_8', 'emb_node2vec_9', 'emb_node2vec_10', 'emb_node2vec_11', 'emb_node2vec_12', 'emb_node2vec_13', 'emb_node2vec_14', 'emb_node2vec_15']


,emb_node2vec_0,emb_node2vec_1,emb_node2vec_2,emb_node2vec_3,emb_node2vec_4,emb_node2vec_5,emb_node2vec_6,emb_node2vec_7,emb_node2vec_8,emb_node2vec_9,emb_node2vec_10,emb_node2vec_11,emb_node2vec_12,emb_node2vec_13,emb_node2vec_14,emb_node2vec_15
0,-0.887847,0.898602,0.012576,-0.714520,0.938697,2.096804,0.044226,-1.160875,0.328630,-1.081705,-0.851846,1.851835,-0.426036,1.006441,0.868429,-1.136214
1,0.375280,-0.608185,1.578555,-0.931551,0.221871,-0.813157,-0.121346,-1.003150,-0.747405,1.148990,-0.167053,0.321360,-0.450833,-0.448637,0.532592,0.161546
2,1.957201,0.978973,2.273713,-1.069282,0.731749,0.566897,0.174864,-1.738916,0.254820,-1.099427,0.685417,0.166507,-2.058650,0.817523,-0.026546,-0.181263


## Step 4 – Load struc2vec embeddings

Struc2Vec captures **structural identity** – nodes with similar local graph structure
get similar embeddings, even if they are far apart in the graph.

The `.emb` file format (Word2Vec output):
- Line 1: `num_nodes num_dimensions`
- Remaining lines: `node_id  dim_1  dim_2  …  dim_16`

In [5]:
def load_struc2vec(path: Path) -> pd.DataFrame:
    """Load struc2vec .emb file, sort by node_id, return embedding columns only."""
    s2v_raw = pd.read_csv(path, sep=" ", skiprows=1, header=None)

    n_dims = s2v_raw.shape[1] - 1  # first column is node_id
    col_names = ["node_id"] + [f"emb_struc2vec_{i}" for i in range(1, n_dims + 1)]
    s2v_raw.columns = col_names

    # Sort by node_id so rows align with features DataFrame
    s2v_raw = s2v_raw.sort_values("node_id").reset_index(drop=True)
    s2v_raw = s2v_raw.drop(columns=["node_id"])
    return s2v_raw

s2v = load_struc2vec(PATH_S2V)

print(f"Shape: {s2v.shape}  (nodes × embedding dimensions)")
s2v.head(3)

Shape: (22470, 16)  (nodes × embedding dimensions)


,emb_struc2vec_1,emb_struc2vec_2,emb_struc2vec_3,emb_struc2vec_4,emb_struc2vec_5,emb_struc2vec_6,emb_struc2vec_7,emb_struc2vec_8,emb_struc2vec_9,emb_struc2vec_10,emb_struc2vec_11,emb_struc2vec_12,emb_struc2vec_13,emb_struc2vec_14,emb_struc2vec_15,emb_struc2vec_16
0,0.423237,-0.143973,0.318151,-0.674688,0.895140,-0.328387,0.886119,-1.706015,0.393010,-0.003067,0.024880,0.654180,-0.459216,0.976516,-1.656966,1.193055
1,0.378943,0.206192,-1.638375,0.641899,2.305374,-1.437089,0.501344,0.157750,0.057912,-0.367381,0.437664,-0.628831,-0.922429,1.508814,-1.529628,-1.321841
2,1.198181,-0.870907,-1.009477,-1.568062,1.930982,-0.728158,0.345341,-1.454881,-2.059695,0.287959,0.257469,1.003040,-2.130869,1.823802,0.662721,1.503481


## Step 5 – Combine features and embeddings

In [6]:
# Concatenate horizontally: node features | node2vec | struc2vec
combined = pd.concat([features, n2v, s2v], axis="columns")

print(f"Combined shape: {combined.shape}")
print(f"Total columns: {len(combined.columns)}")
combined.head(3)

Combined shape: (22470, 55)
Total columns: 55


,node_id,target,participation,in_mod_deg,beta_star,edge_contr,cada,kl,l1,l2,...,emb_struc2vec_7,emb_struc2vec_8,emb_struc2vec_9,emb_struc2vec_10,emb_struc2vec_11,emb_struc2vec_12,emb_struc2vec_13,emb_struc2vec_14,emb_struc2vec_15,emb_struc2vec_16
0,1,False,1.000000,-0.784970,0.996174,1.000000,1.000000,4.436440,1.976324,1.016336,...,0.886119,-1.706015,0.393010,-0.003067,0.024880,0.654180,-0.459216,0.976516,-1.656966,1.193055
1,2,False,0.783737,0.283540,0.826617,0.882353,1.133333,2.996689,1.746428,0.881411,...,0.501344,0.157750,0.057912,-0.367381,0.437664,-0.628831,-0.922429,1.508814,-1.529628,-1.321841
2,3,False,1.000000,0.923698,0.980070,1.000000,1.000000,3.090420,1.909034,0.982675,...,0.345341,-1.454881,-2.059695,0.287959,0.257469,1.003040,-2.130869,1.823802,0.662721,1.503481


## Step 6 – Handle missing values

Nodes without a community assignment have `NaN` in the `target` column.
We drop those rows (they can't be used for classification).
Remaining NaNs in features are filled with 0.

In [7]:
print(f"Rows before NaN drop: {len(combined)}")
print(f"NaNs in 'target': {combined['target'].isna().sum()}")

combined = combined.dropna(subset=["target"]).reset_index(drop=True)

# Fill remaining NaNs with 0 (e.g. eigenvector centrality on isolated nodes)
imputer = SimpleImputer(strategy="constant", fill_value=0)
combined = pd.DataFrame(
    imputer.fit_transform(combined),
    columns=combined.columns
)

print(f"Rows after NaN drop : {len(combined)}")
print(f"Remaining NaNs total: {combined.isna().sum().sum()}")

Rows before NaN drop: 22470
NaNs in 'target': 0
Rows after NaN drop : 22470
Remaining NaNs total: 0


## Step 7 – Summary and save

In [8]:
COMMUNITY_FREE = ["lcc", "bc", "cc", "dc", "ndc", "ec", "eccen", "core"]
COMMUNITY_AWARE = ["participation", "in_mod_deg"]
REGRESSION_TARGETS = ["cada", "edge_contr", "in_mod_deg", "participation",
                      "beta_star", "l1", "l2", "kl", "hd", "l12", "l22", "kl2", "hd2"]
N2V_COLS  = [c for c in combined.columns if c.startswith("emb_node2vec")]
S2V_COLS  = [c for c in combined.columns if c.startswith("emb_struc2vec")]

print(f"Dataset            : {DATASET}")
print(f"Nodes              : {len(combined)}")
print(f"Community-free     : {len(COMMUNITY_FREE)} features")
print(f"Community-aware    : {len(COMMUNITY_AWARE)} features (cheating baseline)")
print(f"Node2Vec dims      : {len(N2V_COLS)}")
print(f"Struc2Vec dims     : {len(S2V_COLS)}")
print(f"Regression targets : {len(REGRESSION_TARGETS)}")
print(f"Classification     : 'target' (binary)")
print(f"\nClass balance:\n{combined['target'].value_counts()}")

combined.to_parquet(PATH_OUTPUT, index=False)
print(f"\nSaved to: {PATH_OUTPUT}")

Dataset            : facebook
Nodes              : 22470
Community-free     : 8 features
Community-aware    : 2 features (cheating baseline)
Node2Vec dims      : 16
Struc2Vec dims     : 16
Regression targets : 13
Classification     : 'target' (binary)

Class balance:
target
0.0    16702
1.0     5768
Name: count, dtype: int64

Saved to: ../data/04_models/facebook_combined.parquet
